# Separating Data Loading, Training, and Inference Code

Machine learning projects often start as a single notebook where everything happens in one place: data loading, preprocessing, feature engineering, model training, evaluation, and sometimes even prediction. This is perfectly acceptable for exploration. It is not acceptable for building reliable systems.

## Overview

As your ML projects grow, mixing data loading, training logic, and inference code in one file leads to:

- Duplication
- Hidden bugs
- Data leakage
- Deployment failures

This lesson focuses on building discipline into your codebase by separating responsibilities clearly and designing your project in a way that supports reproducibility, maintainability, and safe deployment.

The goal is not just cleaner code. The goal is building systems that behave correctly in both training and production environments.

## Why Separation Matters in ML Projects

Unlike traditional scripts, ML workflows have two fundamentally different execution modes:

### Training Mode
The system learns from historical labeled data. It:
- Fits preprocessing steps
- Optimizes model parameters
- Evaluates performance on held-out data
- Saves artifacts

### Inference Mode
The system receives new, unseen data. It:
- Loads previously saved artifacts
- Applies transformations without refitting
- Produces predictions

When these modes are mixed in one file, the most common failures occur:

- Preprocessing gets refit at prediction time
- The model is retrained accidentally during inference
- Data leakage occurs because train/test separation is not respected
- Production prediction logic becomes tightly coupled to training experiments
- Reproducibility becomes impossible

**Separation enforces discipline.** It forces you to think clearly about what happens during training versus what happens during prediction. It prevents accidental refitting. It reduces risk.

## Conceptual Architecture of a Clean ML Workflow

Before writing code, understand the logical boundaries.

A well-structured ML system typically separates:

### Data Loading
Responsible only for reading raw data and returning structured data objects. It does not clean, split, or transform.

### Training
Responsible for preprocessing (fitting), feature engineering, model fitting, evaluation, artifact saving, and experiment logging.

### Inference
Responsible for loading saved artifacts, applying preprocessing transformations (without fitting), and generating predictions on new data.

Each of these responsibilities lives in a separate module or file.

**You should be able to run training without touching inference code. You should be able to run inference without triggering training logic.**

That separation is the design objective.

## Designing the Data Loading Layer

The data loading layer has a narrow responsibility: load raw data into memory in a predictable format.

### What It Should NOT Do
- Perform train/test split
- Fit scalers or encoders
- Modify target variables
- Create derived features

### What It SHOULD Do
- Validate file paths
- Ensure schema consistency
- Preserve original dtypes
- Raise descriptive errors if data is malformed

In [ ]:
# src/data_loader.py

import pandas as pd

def load_data(filepath: str) -> pd.DataFrame:
    """
    Load raw data from CSV file.
    
    Args:
        filepath: Path to the CSV file
    
    Returns:
        pd.DataFrame: Raw dataset
    
    Raises:
        FileNotFoundError: If file does not exist
        ValueError: If dataset is empty
    """
    try:
        df = pd.read_csv(filepath)
    except FileNotFoundError:
        raise FileNotFoundError(f"File not found at path: {filepath}")

    if df.empty:
        raise ValueError("Loaded dataset is empty.")

    return df

## Key Principle: Data Loading

This function is **reusable**. It does not care whether the data is being loaded for training or evaluation. It simply returns the raw dataset.

By isolating loading logic, you prevent hidden transformations from occurring before the train/test split.

## Designing the Training Layer

Training is the **only place** where fitting happens.

This includes:
- Splitting the dataset
- Fitting imputers, encoders, and scalers
- Fitting the model
- Evaluating on held-out data
- Saving artifacts

**Training logic should never be called during inference.**

In [ ]:
# src/train.py

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import joblib
from data_loader import load_data
from preprocessing import build_preprocessing_pipeline

def train_model(data_path: str, model_path: str):
    """
    Train and save a machine learning model.
    
    Args:
        data_path: Path to raw training data
        model_path: Path where trained model will be saved
    """
    # Load raw data
    df = load_data(data_path)

    # Separate features and target
    X = df.drop("target", axis=1)
    y = df["target"]

    # Split data (ONLY during training)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Fit preprocessing pipeline on training data
    pipeline = build_preprocessing_pipeline()
    X_train_processed = pipeline.fit_transform(X_train)
    X_test_processed = pipeline.transform(X_test)

    # Train model
    model = RandomForestClassifier(random_state=42)
    model.fit(X_train_processed, y_train)

    # Save artifacts
    joblib.dump(pipeline, "models/preprocessing.pkl")
    joblib.dump(model, model_path)
    
    print(f"Model saved to {model_path}")

## Key Principles: Training Layer

Notice these critical principles:

1. **Fitting only on training data** — The preprocessing pipeline is fitted only on `X_train`. The test set is never used during fitting.

2. **Train/test separation** — Train and test are split early and processed separately.

3. **Artifacts saved** — After training completes, artifacts are saved for later use.

4. **No inference logic** — This file does not serve predictions. It produces artifacts.

## Designing the Inference Layer

Inference must **never refit anything**.

It must:
- Load saved artifacts
- Validate input schema
- Apply transformations using `transform()`, not `fit_transform()`
- Generate predictions
- Return structured output

In [ ]:
# src/predict.py

import joblib
import pandas as pd

def load_artifacts():
    """
    Load saved preprocessing pipeline and model.
    
    Returns:
        tuple: (preprocessing_pipeline, trained_model)
    """
    pipeline = joblib.load("models/preprocessing.pkl")
    model = joblib.load("models/model.pkl")
    return pipeline, model

def predict(input_data: pd.DataFrame):
    """
    Generate predictions on new data.
    
    Args:
        input_data: pd.DataFrame with features
    
    Returns:
        pd.DataFrame: Predictions and probabilities
    """
    pipeline, model = load_artifacts()

    # Apply preprocessing WITHOUT refitting
    features = pipeline.transform(input_data)
    
    # Generate predictions
    predictions = model.predict(features)
    probabilities = model.predict_proba(features)[:, 1]

    return pd.DataFrame({
        "prediction": predictions,
        "probability": probabilities
    })

## Critical Principle: Inference Layer

**Inference uses `transform()`, never `fit_transform()`.**

This ensures that preprocessing parameters learned during training are reused exactly as-is. This eliminates data leakage and ensures consistency between training and prediction.

## Preventing Common Architectural Mistakes

When separation is not enforced, predictable problems occur.

### Mistake 1: Refitting During Prediction
If preprocessing code is duplicated inside inference logic and uses `fit_transform()`, the model will behave inconsistently between training and production.

### Mistake 2: Hidden Global Variables
If training and prediction share mutable global state, behavior becomes unpredictable.

### Mistake 3: Coupled Experiment Logic
If training scripts import inference logic and vice versa, responsibilities become blurred.

### Mistake 4: Notebook-Centric Design
If everything exists inside one notebook, reproducibility and modular reuse become difficult.

**Separation of concerns is not stylistic preference. It is defensive engineering.**

## File Structure Example

A clean ML project may look like:

```text
project-root/
│
├── data/
│   └── raw/
│
├── models/
│
├── src/
│   ├── data_loader.py
│   ├── preprocessing.py
│   ├── train.py
│   ├── evaluate.py
│   └── predict.py
│
├── requirements.txt
└── README.md
```

Each file has one responsibility:

- Data loading does not train
- Training does not serve
- Inference does not fit

## What This Modularity Enables

This separation allows:

- **Unit testing** individual components
- **Reusing inference** in APIs without triggering training
- **Running training** in batch jobs independently
- **Maintaining reproducibility** across teams
- **Safe deployment** without unexpected model refitting

## Why This Matters in Real Systems

In production ML systems:

- Training may run **once per week**
- Inference may run **thousands of times per minute**

If inference triggers training logic accidentally, your system will fail under load.

If preprocessing refits during prediction, your outputs will **drift silently**.

If data loading logic contains hidden cleaning steps, debugging becomes impossible.

Separation makes each stage independently testable. It makes failures traceable. It allows teams to collaborate without stepping on each other's logic.

**Most ML failures in industry are not algorithmic failures. They are architectural failures.**

## Mental Model to Carry Forward

Keep this model in mind:

**Training produces artifacts. Inference consumes artifacts.**

- Data loading supplies raw inputs
- Preprocessing learns transformations during training and applies them during inference
- The model learns patterns during training and applies them during inference

**These flows must remain structurally independent.**

Once you internalize this, you stop writing notebook-style ML code and start building systems.

## Bonus Content🎁

This section is optional. Learners who want to explore the topics covered so far can utilize the materials provided below.

- Designing Machine Learning Systems
- Google MLOps Architecture Overview
- ML System Design Interview Notes